In [1]:
%load_ext autoreload
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

In [2]:
from ioutils import FileIO
from fileutils import DirInfo, FileInfo
from master import MasterParams, MusicDBPermDir
from pandas import Series, DataFrame, concat
from sys import prefix
from listUtils import getFlatList
from musicdb import PanDBIO
mp    = MasterParams(verbose=True)
io    = FileIO()
mdbpd = MusicDBPermDir()

MasterBasic()
  ==> ModVals:    100
  ==> Project:    pandb
  ==> MusicDB:    musicdb
MasterPaths()
  ==> Raw:        /Volumes/Piggy/Discog
  ==> Mod:        /Volumes/Seagate/Discog
  ==> Sum:        /Users/tgadfort/Music/Discog
MasterMetas()
  ==> Media:      ['Album', 'SingleEP', 'Appearance', 'Technical', 'Mix', 'Bootleg', 'AltMedia', 'Other']
  ==> Metas:      ['Basic', 'Media', 'Genre', 'Bio', 'Link', 'Metric', 'Counts']
  ==> Searches:   ['Name', 'AlbumMedia', 'SingleEPMedia', 'AppearanceMedia', 'TechnicalMedia', 'MixMedia', 'BootlegMedia', 'AltMediaMedia', 'OtherMedia']
MasterDBs()
  ==> DBs:        ['Discogs', 'Spotify', 'LastFM', 'Genius', 'RateYourMusic', 'MetalArchives', 'Deezer', 'AllMusic', 'MusicBrainz', 'AlbumOfTheYear', 'SetListFM', 'Beatport', 'Traxsource', 'MyMixTapez', 'ClassicalArchives']


In [3]:
from lib import beatport
mio   = beatport.MusicDBIO(verbose=False)
webio = beatport.RawWebData()
db    = mio.db
permDBDir = mdbpd.getDBPermPath(db)
print("Saving Perminant {0} DB Data To {1}".format(db, permDBDir.str))

Saving Perminant Beatport DB Data To /Users/tgadfort/opt/anaconda3/envs/py310/pandb/mdblib/Beatport


In [4]:
from base import MusicDBDir, MusicDBData
permDir = MusicDBDir(permDBDir)
localArtists         = MusicDBData(path=permDir, fname="{0}SearchedForLocalArtists".format(db.lower()))
localArtistsReleases = MusicDBData(path=permDir, fname="{0}SearchedForLocalArtistsReleases".format(db.lower()))
localArtistsTracks   = MusicDBData(path=permDir, fname="{0}SearchedForLocalArtistsTracks".format(db.lower()))
masterArtists        = MusicDBData(path=permDir, fname="{0}SearchedForMasterArtists".format(db.lower()))
searchArtists        = mio.data.getSearchArtistData()
knownArtists         = mio.data.getSummaryNameData()
errors               = MusicDBData(path=permDir, fname="{0}SearchedForErrors".format(db.lower()))

In [5]:
##########################################################################################
# Show Summary
##########################################################################################
print("{0} Search Results".format(db))
print("   Local Artists:          {0}".format(len(localArtists.get())))
print("   Local Artists Releases: {0}".format(len(localArtistsReleases.get())))
print("   Local Artists Tracks:   {0}".format(len(localArtistsTracks.get())))
print("   Master Artists:         {0}".format(len(masterArtists.get())))
print("   Errors:                 {0}".format(len(errors.get())))
print("   Search Artists:         {0}".format(searchArtists.shape[0]))
print("   Known IDs:              {0}".format(len(knownArtists)))

Beatport Search Results
   Local Artists:          203325
   Local Artists Releases: 22852
   Local Artists Tracks:   0
   Master Artists:         0
   Errors:                 80
   Search Artists:         203405
   Known IDs:              203187


# Download Artist Data

In [ ]:
mio   = beatport.MusicDBIO(verbose=False,local=True,mkDirs=False)
webio = beatport.RawWebData(debug=False)

In [ ]:
useSearchData = True

if useSearchData is True:
    artistNames      = searchArtists.sort_values(by="Num", ascending=False)
    localArtistsDict = localArtists.get()
    artistNamesToGet = artistNames[~artistNames.index.isin(localArtistsDict.keys())].rename(columns={"Ref": "URL"})

    print("{0} Search Results".format(db))
    print("   Available Names:      {0}".format(len(artistNames)))
    print("   Known Artist Names:   {0}".format(len(localArtistsDict)))
    print("   Artist Names To Get:  {0}".format(len(artistNamesToGet)))
    
#   Artist Names To Get:  39998
#   Artist Names To Get:  33523
#   Artist Names To Get:  12448

In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} ArtistIDs".format(db))
tt = TermTime("tomorrow", "6:50am")
#tt = TermTime("today", "7:00pm")
#tt = TermTime("today", "10:00am")
maxN = 50000000

n  = 0
localArtistsDict  = localArtists.get()
searchedForErrors = errors.get()

for i,(artistID,row) in enumerate(artistNamesToGet.iterrows()):
    artistName = row["Name"]
    artistURL  = row["URL"]
    if localArtistsDict.get(artistID) is not None:
        continue
    if searchedForErrors.get(artistID) is not None:
        continue
        
    response = webio.getArtistData(artistName=artistName, artistURL=artistURL)
    if response is None or len(response) == 0:
        print("Error ==> {0}".format((artistID,artistName)))
        searchedForErrors[artistID] = artistName
        errors.save(data=searchedForErrors)
        webio.sleep(5)
        continue
        
    mio.data.saveRawData(data=response, modval=mio.getModVal(artistID), dbID=artistID)
    localArtistsDict[artistID] = artistName
    webio.sleep(4.0)
    n += 1
        
    if n % 25 == 0:
        print("="*150)
        ts.update(n=n)
        print("Saving {0} {1} Artists Data".format(len(localArtistsDict), db))
        localArtists.save(data=localArtistsDict)
        print("="*150)
        webio.wait(5.0)
        if tt.isFinished():
            break
    
ts.stop()
print("Saving {0} {1} Artists Data".format(len(localArtistsDict), db))
localArtists.save(data=localArtistsDict)
if len(searchedForErrors) > 0:
    print("Saving {0} {1} Errors".format(len(searchedForErrors), db))
    errors.save(data=searchedForErrors)

In [ ]:
localArtists.save(data=localArtistsDict)

In [ ]:
from lib.beatport import moveLocalFiles
moveLocalFiles()

In [ ]:
from utils import PoolIO
pio = PoolIO("Beatport")
pio.parse()
pio.merge()
pio.meta()
pio.sum()
pio.search()

# Download Artist Releases Data

In [6]:
mio   = beatport.MusicDBIO(verbose=False,local=True,mkDirs=False)
webio = beatport.RawWebData(debug=False)

In [45]:
useCountsData = True

if useCountsData is True:
    artistNamesX = DataFrame(mio.data.getSummaryNameData()).join(mio.data.getSummaryRefData()).join(mio.data.getSummaryCountsData())[["Name", "Ref", "Album"]].sort_values(by="Album", ascending=False)
    artistNames = artistNamesX[artistNamesX["Album"] > 0]
    localArtistsReleasesDict = localArtistsReleases.get()
    artistNamesToGet = artistNames[~artistNames.index.isin(localArtistsReleasesDict.keys())].rename(columns={"Ref": "URL"})

    print("# {0} Search Results".format(db))
    print("#   Available Names:      {0}".format(len(artistNamesX)))
    print("#     At Least One Album: {0}".format(len(artistNames)))
    print("#   Known Artist Names:   {0}".format(len(localArtistsReleasesDict)))
    print("#   Artist Names To Get:  {0}".format(len(artistNamesToGet)))
    
#   Artist Names To Get:  139844
#   Artist Names To Get:  128744

# Beatport Search Results
#   Available Names:      203187
#     At Least One Album: 202771
#   Known Artist Names:   76802
#   Artist Names To Get:  125969


In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} ArtistIDs".format(db))
tt = TermTime("tomorrow", "10:50am")
#tt = TermTime("today", "8:00pm")
maxN = 50000000

n  = 0
localArtistsReleasesDict = localArtistsReleases.get()
searchedForErrors = errors.get()

for i,(artistID,row) in enumerate(artistNamesToGet.iterrows()):
    artistName = row["Name"]
    artistURL  = row["URL"]
    if localArtistsReleasesDict.get(artistID) is not None:
        continue
    if searchedForErrors.get(artistID) is not None:
        continue
        
    response = webio.getArtistReleaseData(artistName=artistName, artistURL=artistURL)
    if response is None or len(response) == 0:
        print("Error ==> {0}".format((artistID,artistName)))
        searchedForErrors[artistID] = artistName
        errors.save(data=searchedForErrors)
        webio.sleep(4)
        continue
        
    mio.data.saveRawArtistReleaseData(data=response, modval=mio.getModVal(artistID), dbID=artistID)
    localArtistsReleasesDict[artistID] = artistName
    webio.sleep(4.0)
    n += 1
        
    if n % 25 == 0:
        print("="*150)
        ts.update(n=n)
        print("Saving {0} {1} Artists Data".format(len(localArtistsReleasesDict), db))
        localArtistsReleases.save(data=localArtistsReleasesDict)
        print("="*150)
        webio.wait(10.0)
        if tt.isFinished():
            break
    
ts.stop()
print("Saving {0} {1} Artists Data".format(len(localArtistsReleasesDict), db))
localArtistsReleases.save(data=localArtistsReleasesDict)
if len(searchedForErrors) > 0:
    print("Saving {0} {1} Errors".format(len(searchedForErrors), db))
    errors.save(data=searchedForErrors)

Process [Getting Beatport ArtistIDs] Start    ==> Time Is 2022-05-19 20:09:48
========================= termTime(day=tomorrow,time=10:50am) =========================
   ====> Terminate Time Set To 2022-05-20 10:50:00 <====
   ====> Will Terminate Process 14 Hours and 40 Minutes From Now
Getting Data For Mind the Gaap (638914)                            True
Getting Data For Jolly (82514)                                     True
Getting Data For Eric Clap (835914)                                True
Getting Data For Nicola Hitchcock (16414)                          True
Getting Data For Christopher Erre (115714)                         True
Getting Data For Al-Fernandez (225814)                             True
Getting Data For Mark Coppi (964114)                               True
Getting Data For Adam Madam (293314)                               True
Getting Data For Double Amp (270114)                               True
Getting Data For Isolee (7214)                                  

# Download Artist Tracks Data

In [ ]:
useCountsData = True

if useCountsData is True:
    artistNamesX = DataFrame(mio.data.getSummaryNameData()).join(mio.data.getSummaryRefData()).join(mio.data.getSummaryCountsData())[["Name", "Ref", "SingleEP"]].sort_values(by="SingleEP", ascending=False)
    artistNames = artistNamesX[artistNamesX["SingleEP"] > 0]
    localArtistsReleasesDict = localArtistsReleases.get()
    artistNamesToGet = artistNames[~artistNames.index.isin(localArtistsReleasesDict.keys())].rename(columns={"Ref": "URL"})

    print("{0} Search Results".format(db))
    print("   Available Names:      {0}".format(len(artistNamesX)))
    print("     At Least One Track: {0}".format(len(artistNames)))
    print("   Known Artist Names:   {0}".format(len(localArtistsReleasesDict)))
    print("   Artist Names To Get:  {0}".format(len(artistNamesToGet)))
    
#   Artist Names To Get:  39998
#   Artist Names To Get:  33523
#   Artist Names To Get:  12448

# Create Starter

In [ ]:
from collections import Counter
from ioutils import FileIO, HTMLIO
io = FileIO()
hio = HTMLIO()
cntr = Counter()
refs = []
for modVal in range(100):
    for ifile in mio.dir.getRawModValDataDir(modVal).glob("*.p"):
        bsdata = hio.get(io.get(ifile))
        refs  += [(ref.attrs['data-artist'],ref.attrs['href'],ref.text) for ref in bsdata.findAll('a') if ref.attrs.get('data-artist') is not None]
    if (modVal+1) % 5 == 0:
        print(modVal,'\t',len(refs))

In [ ]:
df = DataFrame(refs)
N  = df[0].value_counts()
N.name = "Num"
df = df.drop_duplicates()
df.index = df[0]
df.index.name = None
df = df.drop([0], axis=1)
df.columns = ["Ref", "Name"]
df = df.join(N)
df.sort_values(by='Num', ascending=False)

In [ ]:
mio.data.saveSearchArtistData(data=df)